In [78]:
import pandas as pd
import datetime
import os
import json
import altair as alt

df = pd.read_csv("Gesamtdatensatz.csv")

image_paths = ["src/assets/clear-day.png","src/assets/clear-night.png", "src/assets/cloudy.png", "src/assets/fog.png", "src/assets/partly-cloudy-day.png", "src/assets/partly-cloudy-night.png", "src/assets/rain.png", "src/assets/snow.png"]
path = {}

for image_path in image_paths:
    # Schlüssel aus Dateiname extrahieren (ohne Ordner und ohne .png)
    key = os.path.splitext(os.path.basename(image_path))[0]
    # Pfad in das Dictionary einfügen
    path[key] = image_path



# Add columns to table
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['pedestrian_grey'] = df[['ltr_pedestrians_count', 'rtl_pedestrians_count']].min(axis=1)
df['pedestrian_diff'] = ((df['ltr_pedestrians_count'] - df['rtl_pedestrians_count'])**2)**0.5
df['weather_icon'] = df['weather_condition'].map(path)
df['max_val'] = (
    df.groupby(['date'])[['ltr_pedestrians_count', 'rtl_pedestrians_count']]
      .transform('max')      # max per column per date
      .max(axis=1)           # max across the two columns
)+20

f_time_loc = df[(df['timestamp'].dt.date == datetime.date(2022, 1, 18)) & (df['location_name'] == 'Bahnhofstrasse (Mitte)')]
f_time_loc.to_json('time_loc_data.json', orient='records', indent=2)
#f_time = df[(df['timestamp'].dt.date == datetime.date(2021, 10, 15))]
#f_time.to_json('time_data.json', orient='records', indent=2)
#df.to_json('fulldata.json', orient='records', indent=2)

f_time_loc.head(n=5)

,timestamp,location_id,location_name,ltr_label,rtl_label,weather_condition,temperature,pedestrians_count,unverified,collection_type,...,zone_99_ltr_pedestrians_count,zone_99_rtl_pedestrians_count,zone_99_adult_pedestrians_count,zone_99_child_pedestrians_count,date,hour,pedestrian_grey,pedestrian_diff,weather_icon,max_val
10664,2022-01-18 00:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-0.75,10,False,measured,...,NaN,NaN,NaN,NaN,2022-01-18,0,4,2.0,src/assets/partly-cloudy-night.png,1319
10668,2022-01-18 01:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-1.27,16,False,measured,...,NaN,NaN,NaN,NaN,2022-01-18,1,6,4.0,src/assets/partly-cloudy-night.png,1319
10672,2022-01-18 02:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-1.47,5,False,measured,...,NaN,NaN,NaN,NaN,2022-01-18,2,2,1.0,src/assets/partly-cloudy-night.png,1319
10676,2022-01-18 03:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-1.84,10,False,measured,...,NaN,NaN,NaN,NaN,2022-01-18,3,4,2.0,src/assets/partly-cloudy-night.png,1319
10680,2022-01-18 04:00:00+00:00,329,Bahnhofstrasse (Mitte),Hauptbahnhof,Bürkliplatz,partly-cloudy-night,-2.43,96,False,measured,...,NaN,NaN,NaN,NaN,2022-01-18,4,34,28.0,src/assets/partly-cloudy-night.png,1319


In [79]:


#data = pd.read_csv("Gesamtdatensatz.csv")
#data = pd.read_json("time_loc_data.json")
data = 'time_loc_data.json'

# Initialize Basic Chart
base = alt.Chart(data).add_params().properties(width=400, height=24*30)

# Build Temperature chart for Background
inv = base.encode(
    alt.Y('hour:T').axis(None),
    alt.X('max_val:Q',axis=alt.Axis(title=None)), 
    ).mark_rect(
        opacity=0,
        height=0)


temp_left = base.encode(
    alt.Y('hour:T').axis(None), 
    alt.Color('temperature:Q')
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('temperature:Q', 
            title='Temperatur [°C]:', 
            format=".1f")]).mark_rect(
        height=30).interactive()

temp_right = base.encode(
    alt.Y('hour:O').axis(None),
    alt.Color('temperature:Q')
        #.bin(maxbins=10, extent=[-10,35])
        .scale(scheme='blueorange', domain=[-10,35], type="linear"),
    tooltip=[alt.Tooltip('temperature:Q',
            title='Temperatur [°C]:',
            format=".1f")]).mark_rect(
            height=30).interactive()

# Build Pedestrian-count chart
left = (base.transform_calculate(
        tooltip_title="datum.pedestrian_diff + ' Passanten mehr in Richtung ' + datum.rtl_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('ltr_pedestrians_count:Q', axis=alt.Axis(title=None)),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
                color='#6A11B3',
                height=20).interactive()
)                

right = (base.transform_calculate(
        tooltip_title="datum.pedestrian_diff + ' Passanten mehr in Richtung ' + datum.ltr_label")
    .encode(
    alt.Y('hour:O').axis(None), 
    alt.X('rtl_pedestrians_count:Q', axis=alt.Axis(title=None)),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color='#6A11B3', 
            height=20).interactive()
)
# Build Pedestrian-even count chart
left_g = (base.transform_calculate(
        tooltip_title="'Anzahl Passanten in Richtung ' + datum.rtl_label + ': ' + datum.ltr_pedestrians_count")
    .encode(
    alt.Y('hour:O').axis(None),
    alt.X('sum(pedestrian_grey):Q', axis=alt.Axis(title=None))
        .sort('descending'),
    tooltip=[alt.Tooltip('tooltip_title:N', title=' '),]).mark_bar(
            color="#8950B8",
            height=20)
)            

right_g = (base.transform_calculate(
        tooltip_title="'Anzahl Passanten in Richtung ' + datum.ltr_label + ': ' + datum.rtl_pedestrians_count")
    .encode(
        alt.Y('hour:O').axis(None),
        alt.X('sum(pedestrian_grey):Q', axis=alt.Axis(title=None)),
        tooltip=[alt.Tooltip('tooltip_title:N', title=' '),])
    .mark_bar(color='#8950B8', height=20)
)

# weather icon chart
weather = base.transform_aggregate(
    weather_icon='max(weather_icon)',
    groupby=['hour', 'location_id']
).encode(
    alt.Y('hour:O').axis(None),
    url='weather_icon:N'
).mark_image(width=25, height=25).properties(width=30)

# Build middle chart (legend)
middle = base.transform_aggregate(groupby=['hour', 'location_id']).encode(
    alt.Y('hour:O').axis(None),
    alt.Text('hour:O')).mark_text(
        fontSize=15,
        font='Bahnschrift').properties(width=20)


# Layer Charts
left_chart = alt.layer(temp_left, left, left_g, inv)
right_chart = alt.layer(temp_right, right, right_g, inv)

# Concatenate Charts
main_chart = alt.concat(weather, left_chart, middle, right_chart, spacing = 5,).configure_view(
    stroke=None,
).configure_legend(direction='vertical', labelAlign='left', orient='right', gradientLength=720, gradientThickness=25, labelFont='Bahnschrift',
                   labelFontSize=12, titleOrient='right', titleFont='Bahnschrift', titleFontSize=20, titleFontWeight=200)

spec = main_chart.to_dict()

with open("chart.json", "w") as f:
    json.dump(spec, f, indent=2)

main_chart
#https://altair-viz.github.io/user_guide/marks/image.html


c:\Users\jonat\miniconda3\envs\3050WID\Lib\site-packages\IPython\core\interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.ConcatChart(...)